In [ ]:
!pip install transformers==5.0.0 torch==2.10.0 scikit-learn==1.6.1 scipy==1.16.3 gensim==4.4.0 -q

In [ ]:
import os
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score
from transformers import AutoTokenizer, AutoModel
import torch
from gensim.models import Word2Vec, KeyedVectors
import gensim.downloader as api
import huggingface_hub

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

feature_extraction

In [ ]:
# Word2Vec Indonesia (https://github.com/deryrahman/word2vec-bahasa-indonesia)
print("Loading Word2Vec ID...")
w2v_id_model = Word2Vec.load("/content/drive/MyDrive/riset_teks/models/word2vec_id/idwiki_word2vec_300.model")
w2v_id = w2v_id_model.wv
print(f"Word2Vec ID | Vocab: {len(w2v_id):,} | Dim: {w2v_id.vector_size}")

# Word2Vec Inggris (Google News 300)
print("Loading Word2Vec EN...")
w2v_en = api.load("word2vec-google-news-300")
print(f" Word2Vec EN | Vocab: {len(w2v_en):,} | Dim: {w2v_en.vector_size}")

# BERT
print("Loading IndoBERT...")
bert_id_tokenizer = AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p1")
bert_id_model = AutoModel.from_pretrained("indobenchmark/indobert-base-p1").cuda()

print("Loading BiomedBERT...")
bert_en_tokenizer = AutoTokenizer.from_pretrained("microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract")
bert_en_model = AutoModel.from_pretrained("microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract").cuda()

bert_id_model.eval()
bert_en_model.eval()
print("BERT models loaded")

In [ ]:
# TF-IDF
def extract_tfidf(train_texts, test_texts):
    vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        max_features=20000,
        sublinear_tf=True,
        lowercase=False,
    )
    X_train = vectorizer.fit_transform(train_texts)
    X_test  = vectorizer.transform(test_texts)
    return X_train, X_test, vectorizer
# Word2Vec
def text_to_w2v(texts, w2v_model):
    vectors = []
    for text in texts:
        tokens = str(text).split()
        vecs = [w2v_model[t] for t in tokens if t in w2v_model.key_to_index]
        if len(vecs) == 0:
            vectors.append(np.zeros(w2v_model.vector_size))
        else:
            vectors.append(np.mean(vecs, axis=0))
    return np.array(vectors)

# BERT
def text_to_bert(texts, tokenizer, model, batch_size=32, max_length=512):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = list(texts[i:i+batch_size])
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        )
        encoded = {k: v.cuda() for k, v in encoded.items()}
        with torch.no_grad():
            output = model(**encoded)
        embeddings = output.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(embeddings)
        if (i // batch_size) % 10 == 0:
            print(f"    BERT batch {i//batch_size + 1}/{(len(texts)-1)//batch_size + 1}", end='\r')
    return np.vstack(all_embeddings)

In [ ]:
def print_feature_info(X_train, X_test, method, dataset, level, fold):
    print(f"\n  [{method}] {dataset} | Level {level} | Fold {fold}")
    print(f"  Train shape : {X_train.shape}")
    print(f"  Test shape  : {X_test.shape}")
    if sp.issparse(X_train):
        sparsity = 1 - X_train.nnz / (X_train.shape[0] * X_train.shape[1])
        print(f"  Sparsity    : {sparsity:.4%}")
        print(f"  Non-zero    : {X_train.nnz:,}")
    else:
        print(f"  Dimensi     : {X_train.shape[1]} (dense)")

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, classification_report)

BASE_FEAT = '/content/drive/MyDrive/riset_teks/features'
BASE_RESULT = '/content/drive/MyDrive/riset_teks/results'

def save_features(X_train, y_train, X_test, y_test, dataset, method, level, fold):
    path = f"{BASE_FEAT}/{dataset}/{method}/level{level}"
    os.makedirs(path, exist_ok=True)

    if sp.issparse(X_train):
        sp.save_npz(f"{path}/fold{fold}_train.npz", X_train)
        sp.save_npz(f"{path}/fold{fold}_test.npz", X_test)
    else:
        np.save(f"{path}/fold{fold}_train_X.npy", X_train)
        np.save(f"{path}/fold{fold}_test_X.npy", X_test)

    np.save(f"{path}/fold{fold}_train_y.npy", y_train)
    np.save(f"{path}/fold{fold}_test_y.npy", y_test)


def run_svm_and_save(X_train, y_train, X_test, y_test, dataset, method, level, fold):
    clf = LinearSVC(
        C=1.0,
        class_weight='balanced',
        max_iter=5000,
        random_state=42,
        dual='auto'
    )
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    metrics = {
        'accuracy'  : accuracy_score(y_test, y_pred),
        'precision' : precision_score(y_test, y_pred, average='macro', zero_division=0),
        'recall'    : recall_score(y_test, y_pred, average='macro', zero_division=0),
        'f1'        : f1_score(y_test, y_pred, average='macro', zero_division=0),
    }

    report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    report_df = pd.DataFrame(report).transpose()

    path = f"{BASE_RESULT}/{dataset}/{method}/level{level}"
    os.makedirs(path, exist_ok=True)
    report_df.to_csv(f"{path}/fold{fold}_report.csv")

    return metrics

training_evaluation

In [ ]:
DATASETS = {
    'dataset1': {'lang': 'id'},
    'dataset2': {'lang': 'id'},
    'dataset3': {'lang': 'en'},
}

BASE_PREP   = '/content/drive/MyDrive/riset_teks/preprocessed'
BASE_FEAT   = '/content/drive/MyDrive/riset_teks/features'
BASE_RESULT = '/content/drive/MyDrive/riset_teks/results'

summary_rows = []

for dataset, config in DATASETS.items():
    lang = config['lang']


    if lang == 'id':
        w2v_model      = w2v_id
        bert_tokenizer = bert_id_tokenizer
        bert_model_use = bert_id_model
        bert_en_model.cpu()
        torch.cuda.empty_cache()
        bert_id_model.cuda()
    else:
        w2v_model      = w2v_en
        bert_tokenizer = bert_en_tokenizer
        bert_model_use = bert_en_model
        bert_id_model.cpu()
        torch.cuda.empty_cache()
        bert_en_model.cuda()

    print(f"\n{'='*60}")
    print(f"DATASET: {dataset} ({lang.upper()})")
    print(f"{'='*60}")

    for level in range(5):
        print(f"\n  ── Level {level} ──")

        for fold in range(1, 6):
            print(f"    Fold {fold}...")

            # Load data
            train_df = pd.read_csv(f"{BASE_PREP}/{dataset}/level{level}/fold{fold}_train.csv")
            test_df  = pd.read_csv(f"{BASE_PREP}/{dataset}/level{level}/fold{fold}_test.csv")

            train_texts = train_df['text'].fillna('').values
            test_texts  = test_df['text'].fillna('').values
            y_train     = train_df['class'].values
            y_test      = test_df['class'].values

            # TF-IDF
            X_train_tfidf, X_test_tfidf, vec = extract_tfidf(train_texts, test_texts)
            print_feature_info(X_train_tfidf, X_test_tfidf, 'TF-IDF', dataset, level, fold)
            save_features(X_train_tfidf, y_train, X_test_tfidf, y_test, dataset, 'tfidf', level, fold)
            m_tfidf = run_svm_and_save(X_train_tfidf, y_train, X_test_tfidf, y_test, dataset, 'tfidf', level, fold)

            # Word2Vec
            X_train_w2v = text_to_w2v(train_texts, w2v_model)
            X_test_w2v  = text_to_w2v(test_texts,  w2v_model)
            print_feature_info(X_train_w2v, X_test_w2v, 'Word2Vec', dataset, level, fold)
            save_features(X_train_w2v, y_train, X_test_w2v, y_test, dataset, 'word2vec', level, fold)
            m_w2v = run_svm_and_save(X_train_w2v, y_train, X_test_w2v, y_test, dataset, 'word2vec', level, fold)

            # BERT
            X_train_bert = text_to_bert(train_texts, bert_tokenizer, bert_model_use)
            X_test_bert  = text_to_bert(test_texts,  bert_tokenizer, bert_model_use)
            print_feature_info(X_train_bert, X_test_bert, 'BERT', dataset, level, fold)
            save_features(X_train_bert, y_train, X_test_bert, y_test, dataset, 'bert', level, fold)
            m_bert = run_svm_and_save(X_train_bert, y_train, X_test_bert, y_test, dataset, 'bert', level, fold)

            # save to summary
            summary_rows.append({
                'dataset': dataset, 'level': level, 'fold': fold,
                'tfidf_acc' : m_tfidf['accuracy'],  'tfidf_prec': m_tfidf['precision'],
                'tfidf_rec' : m_tfidf['recall'],    'tfidf_f1'  : m_tfidf['f1'],
                'w2v_acc'   : m_w2v['accuracy'],    'w2v_prec'  : m_w2v['precision'],
                'w2v_rec'   : m_w2v['recall'],      'w2v_f1'    : m_w2v['f1'],
                'bert_acc'  : m_bert['accuracy'],   'bert_prec' : m_bert['precision'],
                'bert_rec'  : m_bert['recall'],     'bert_f1'   : m_bert['f1'],
            })

        print(f"  Level {level} done!")

    print(f"\n {dataset} done!")

print("\n completed!")

In [ ]:
summary_df = pd.DataFrame(summary_rows)

os.makedirs(BASE_RESULT, exist_ok=True)
summary_df.to_csv(f"{BASE_RESULT}/summary_all_folds.csv", index=False)

methods      = ['tfidf',   'w2v',      'bert']
method_label = ['TF-IDF',  'Word2Vec', 'BERT']
metrics      = ['acc',     'prec',     'rec',      'f1']
metric_label = ['Accuracy','Precision','Recall',   'F1-Score']

rows = []
for (dataset, level), grp in summary_df.groupby(['dataset', 'level']):
    row = {'dataset': dataset, 'level': level}
    for method in methods:
        for m in metrics:
            col = f"{method}_{m}"
            row[f"{col}_mean"] = round(grp[col].mean(), 4)
            row[f"{col}_std"]  = round(grp[col].std(),  4)
    rows.append(row)

result_df = pd.DataFrame(rows)
result_df.to_csv(f"{BASE_RESULT}/summary_mean_std.csv", index=False)

for dataset in ['dataset1', 'dataset2', 'dataset3']:
    print(f"\n{'='*70}")
    print(f"  DATASET: {dataset}")
    print(f"{'='*70}")
    sub = result_df[result_df['dataset'] == dataset].set_index('level')

    for method, mlabel in zip(methods, method_label):
        print(f"\n  [{mlabel}]")
        print(f"  {'Level':<8}", end="")
        for ml in metric_label:
            print(f"  {ml:<22}", end="")
        print()
        print(f"  {'-'*100}")

        for lvl in range(5):
            print(f"  {lvl:<8}", end="")
            for m in metrics:
                mean = sub.loc[lvl, f"{method}_{m}_mean"]
                std  = sub.loc[lvl, f"{method}_{m}_std"]
                print(f"  {mean:.4f} ± {std:.4f}      ", end="")
            print()

print(f"\n Summary saved in {BASE_RESULT}")